# CSE 164 A100 Supervised Multi-Task Runs

This notebook is for Google Colab A100 runs. It keeps the same Drive data layout as the existing Colab notebook:

```text
/content/drive/MyDrive/CSE164FinalProject/raw/
|-- metadata/
|-- test/
|-- train_labeled/
|-- train_seg/
|-- train_unlabeled/
`-- val/
```

Outputs are written to `/content/drive/MyDrive/CSE164FinalProject/outputs/` so checkpoints and submissions survive runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

## Repository and Paths

If the repo is not already in `/content/CSE-164FinalProject`, upload it, clone it, or copy it from Drive before continuing.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/CSE-164FinalProject')
DRIVE_PROJECT = Path('/content/drive/MyDrive/CSE164FinalProject')
DATA_ROOT = DRIVE_PROJECT / 'raw'
DRIVE_OUTPUTS = DRIVE_PROJECT / 'outputs'

assert REPO_DIR.exists(), f'Missing repo at {REPO_DIR}'
assert DATA_ROOT.exists(), f'Missing data at {DATA_ROOT}'
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
print('repo:', REPO_DIR)
print('data:', DATA_ROOT)
print('drive outputs:', DRIVE_OUTPUTS)

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q kaggle

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
import torch
print(torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Smoke Test

Runs one tiny ResNet-50 multi-task epoch to verify the notebook, data paths, checkpoint writing, and validation reload path.

In [ ]:
!python -m src.training.train_multitask \
  --data-root "$DATA_ROOT" \
  --architecture resnet50 \
  --stage joint \
  --epochs 1 \
  --warmup-epochs 0 \
  --image-size 128 \
  --num-segmentation-classes 1 \
  --seg-batch-size 1 \
  --cls-batch-size 2 \
  --val-batch-size 1 \
  --num-workers 2 \
  --learning-rate 1e-4 \
  --min-learning-rate 1e-6 \
  --weight-decay 1e-4 \
  --label-smoothing 0.1 \
  --segmentation-loss-weight 1.0 \
  --dice-loss-weight 1.0 \
  --seg-classification-loss-weight 1.0 \
  --cls-loss-weight 1.0 \
  --gradient-clip 1.0 \
  --ema-decay 0.9998 \
  --weighted-combined-sampling \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --max-seg-samples 2 \
  --max-cls-samples 4 \
  --max-val-samples 2 \
  --no-random-crop \
  --checkpoint-dir "$DRIVE_OUTPUTS/checkpoints/resnet50_smoke"

## Serious ResNet-50 Supervised Run

This trains ResNet-50 from scratch with shared encoder, U-Net binary decoder, mask-guided classifier pooling, EMA, warmup plus cosine LR, and weighted supervised sampling.

In [ ]:
!python -m src.training.train_multitask \
  --data-root "$DATA_ROOT" \
  --architecture resnet50 \
  --stage joint \
  --epochs 100 \
  --warmup-epochs 2 \
  --image-size 384 \
  --num-segmentation-classes 1 \
  --seg-batch-size 4 \
  --cls-batch-size 32 \
  --val-batch-size 4 \
  --num-workers 8 \
  --learning-rate 1e-3 \
  --min-learning-rate 1e-6 \
  --weight-decay 5e-2 \
  --label-smoothing 0.1 \
  --segmentation-loss-weight 1.0 \
  --dice-loss-weight 1.0 \
  --seg-classification-loss-weight 1.0 \
  --cls-loss-weight 1.0 \
  --gradient-clip 5.0 \
  --ema-decay 0.9998 \
  --weighted-combined-sampling \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --validation-threshold 0.50 \
  --checkpoint-dir "$DRIVE_OUTPUTS/checkpoints/resnet50_multitask_384"

## Threshold Sweep

Run after training finishes. Pick the threshold with the best automated validation score.

In [ ]:
!python -m src.training.tune_multitask_threshold \
  --seg-checkpoint "$DRIVE_OUTPUTS/checkpoints/resnet50_multitask_384/best_multitask.pt" \
  --data-root "$DATA_ROOT" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 8 \
  --tta hflip \
  --thresholds 0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75

## Generate Submission

Replace `BEST_THRESHOLD` with the best threshold from the sweep.

In [ ]:
BEST_THRESHOLD = 0.50

In [ ]:
!python -m src.training.predict_multitask \
  --checkpoint "$DRIVE_OUTPUTS/checkpoints/resnet50_multitask_384/best_multitask.pt" \
  --data-root "$DATA_ROOT" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 8 \
  --tta hflip \
  --seg-threshold "$BEST_THRESHOLD" \
  --output "$DRIVE_OUTPUTS/submissions/resnet50_multitask_384_tta.csv"

In [ ]:
!python starter/validate_submission_csv.py \
  --submission "$DRIVE_OUTPUTS/submissions/resnet50_multitask_384_tta.csv" \
  --data-root "$DATA_ROOT" \
  --split test